In [1]:
import pandas as pd
df = pd.read_csv('/content/sample_data/fraud_clauses_dataset.csv')
df.head()

,clause_text,risk_index,label,probability_of_risk,category,source
0,Husband central door. Example point program hi...,medium,0,0.50,termination_rights,synthetic_template
1,Three including heavy majority week energy. Gu...,medium,0,0.64,third_party_sharing,semi_random_generated
2,Officer its by never future account bed. Inter...,high,1,0.97,consent_clarity,semi_random_generated
3,What small south minute weight eat officer. Yo...,medium,0,0.51,interest_rates,synthetic_template
4,Already left growth with condition. Personal d...,low,0,0.00,payment_terms,synthetic_template


In [2]:
!pip install transformers datasets

from transformers import AutoTokenizer, AutoModelForMaskedLM
from datasets import Dataset

df_model = df[["clause_text", "label"]].copy()
dataset = Dataset.from_pandas(df_model)

tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")

def tokenize_function(example):
    return tokenizer(
        example["clause_text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tknzed_dset = dataset.map(tokenize_function, batched=True)
tknzed_dset = tknzed_dset.rename_column("label", "labels")
tknzed_dset = tknzed_dset.remove_columns(["clause_text"])
tknzed_dset.set_format("torch")
tknzed_dset[0]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

{'labels': tensor(0),
 'input_ids': tensor([  101,  3129,  2430,  2341,  1012,  2742,  2391,  2565,  2152,  2082,
          2101,  1012,  2057,  2089,  3715,  1037, 10509, 25761,  7408,  2044,
          2382,  2420,  1997, 27118,  6593,  7730,  1012,   102,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,  

In [3]:
df.head()

,clause_text,risk_index,label,probability_of_risk,category,source
0,Husband central door. Example point program hi...,medium,0,0.50,termination_rights,synthetic_template
1,Three including heavy majority week energy. Gu...,medium,0,0.64,third_party_sharing,semi_random_generated
2,Officer its by never future account bed. Inter...,high,1,0.97,consent_clarity,semi_random_generated
3,What small south minute weight eat officer. Yo...,medium,0,0.51,interest_rates,synthetic_template
4,Already left growth with condition. Personal d...,low,0,0.00,payment_terms,synthetic_template


In [8]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import DatasetDict

model = AutoModelForSequenceClassification.from_pretrained("google-bert/bert-base-uncased", num_labels=2)
split_dataset = tknzed_dset.train_test_split(test_size=0.1, seed=42)

print("Train set size:", len(split_dataset["train"]))
print("Test set size:", len(split_dataset["test"]))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Train set size: 4500
Test set size: 500


In [15]:
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

# Configure training arguments specifically for TPU compatibility, explicitly using standard unfused adamw_torch
training_args = TrainingArguments(
    output_dir="./bert_fraud_detector",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    optim="adamw_torch"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split_dataset["train"],
    eval_dataset=split_dataset["test"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [14]:
trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.000315,0.000190,1.000000,1.000000,1.000000,1.000000
2,0.000135,0.000083,1.000000,1.000000,1.000000,1.000000
3,0.000092,0.000057,1.000000,1.000000,1.000000,1.000000
4,0.000085,0.000051,1.000000,1.000000,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=1128, training_loss=0.012787989106292239, metrics={'train_runtime': 445.2782, 'train_samples_per_second': 40.532, 'train_steps_per_second': 2.533, 'total_flos': 1183999749120000.0, 'train_loss': 0.012787989106292239, 'epoch': 4.0})

In [17]:
# Move model back to CPU to ensure valid host storage pointers for saving
model.to('cpu')

model.save_pretrained("bert_fraud_classifier")
tokenizer.save_pretrained("bert_fraud_classifier")

import shutil
shutil.make_archive("bert_fraud_classifier", 'zip', "bert_fraud_classifier")

from google.colab import files
files.download("bert_fraud_classifier.zip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

def predict_clause_risk(text):
    device = next(model.parameters()).device
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Run inference
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = F.softmax(logits, dim=1).squeeze().tolist()

    # Interpret result
    label = int(torch.argmax(logits, dim=1).item())
    label_name = "RISKY" if label == 1 else "SAFE"
    confidence = round(probs[label] * 100, 2)

    return {
        "text": text,
        "predicted_label": label,
        "label_meaning": label_name,
        "confidence": f"{confidence}%",
        "probabilities": {
            "SAFE (0)": round(probs[0], 4),
            "RISKY (1)": round(probs[1], 4)
        }
    }


In [19]:
test_clauses = [
    "Late payments will incur a 20% recurring monthly penalty.",
    "We reserve the right to terminate your access at any time without notice.",
    "Personal data is stored securely and will only be used for service delivery.",
    "Interest rates are capped at 7.5% per annum as per RBI norms.",
    "This agreement is binding upon signature, and you forfeit all refund rights.",
    "User activity may be shared with unknown third parties.",
    "You may cancel anytime without penalty.",
]

for clause in test_clauses:
    result = predict_clause_risk(clause)
    print("\n---")
    for k, v in result.items():
        print(f"{k}: {v}")



---
text: Late payments will incur a 20% recurring monthly penalty.
predicted_label: 1
label_meaning: RISKY
confidence: 99.98%
probabilities: {'SAFE (0)': 0.0002, 'RISKY (1)': 0.9998}

---
text: We reserve the right to terminate your access at any time without notice.
predicted_label: 1
label_meaning: RISKY
confidence: 99.96%
probabilities: {'SAFE (0)': 0.0004, 'RISKY (1)': 0.9996}

---
text: Personal data is stored securely and will only be used for service delivery.
predicted_label: 0
label_meaning: SAFE
confidence: 99.97%
probabilities: {'SAFE (0)': 0.9997, 'RISKY (1)': 0.0003}

---
text: Interest rates are capped at 7.5% per annum as per RBI norms.
predicted_label: 0
label_meaning: SAFE
confidence: 99.02%
probabilities: {'SAFE (0)': 0.9902, 'RISKY (1)': 0.0098}

---
text: This agreement is binding upon signature, and you forfeit all refund rights.
predicted_label: 0
label_meaning: SAFE
confidence: 99.74%
probabilities: {'SAFE (0)': 0.9974, 'RISKY (1)': 0.0026}

---
text: User acti